<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Deep_learning_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cell 1 — GPU check + setup

In [1]:
!nvidia-smi
import os
from pathlib import Path

ROOT = Path("/content/av_perception")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

Tue Feb  3 16:18:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Cell 2 — Install YOLOv8 + utils

In [2]:
!pip -q install ultralytics opencv-python imageio imageio-ffmpeg

from ultralytics import YOLO

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.5/934.5 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.5 MB/s eta 0:00:

### Cell 3 — Download dataset (YOLO-ready, no login)

In [3]:
# Public YOLO-format road dataset (mirrored)
!wget -q https://public.roboflow.com/datasets/road-sign-detection/1/download/yolov5 -O road_data.zip
!unzip -q road_data.zip -d road_data

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


Preparing split 'train' in '/root/fiftyone/bdd100k/train'


INFO:fiftyone.zoo.datasets:Preparing split 'train' in '/root/fiftyone/bdd100k/train'


OSError: 

You must provide a `source_dir` in order to load the BDD100K dataset.

You must download the source files for BDD100K dataset manually.

Run `fiftyone zoo datasets info bdd100k` for more information

### Cell 4 — Inspect dataset

In [ ]:
import yaml

with open("road_data/data.yaml") as f:
    data_yaml = yaml.safe_load(f)

data_yaml

### Cell 5 — Train YOLOv8 (T4-friendly)

In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="road_data/data.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    name="road_detection",
)

### Cell 6 — Validate

In [ ]:
model = YOLO("runs/detect/road_detection/weights/best.pt")
model.val(data="road_data/data.yaml")

### Cell 7 — Download demo driving video

In [ ]:
!wget -q -O traffic.mp4 https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4

### Cell 8 — Detection + tracking + TTC overlay (AV logic)

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture("traffic.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    "traffic_ttc.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H),
)

FOCAL = 700
REAL_HEIGHT = {"person": 1.7, "car": 1.5}
track_hist = {}
TTC_THRESH = 2.0

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t = frame_id / fps
    results = model.track(frame, persist=True, conf=0.25, verbose=False)[0]

    if results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids   = results.boxes.id.cpu().numpy()
        clss  = results.boxes.cls.cpu().numpy().astype(int)

        for box, tid, cls in zip(boxes, ids, clss):
            x1,y1,x2,y2 = map(int, box)
            label = model.names[cls]

            h = max(1, y2 - y1)
            dist = (REAL_HEIGHT.get(label,1.6)*FOCAL)/h

            ttc = None
            if tid in track_hist:
                d_prev, t_prev = track_hist[tid]
                v_rel = (d_prev - dist)/(t - t_prev + 1e-3)
                if v_rel > 0:
                    ttc = dist/v_rel

            track_hist[tid] = (dist,t)

            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,255),2)
            txt = f"{label} id={int(tid)} d~{dist:.1f}m"
            if ttc and ttc < 10:
                txt += f" TTC~{ttc:.1f}s"
            cv2.putText(frame,txt,(x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,255),2)

            if ttc and ttc < TTC_THRESH:
                cv2.putText(frame,"NEAR MISS!",
                            (20,40),
                            cv2.FONT_HERSHEY_SIMPLEX,1.1,(0,0,255),3)

    out.write(frame)
    frame_id += 1

cap.release()
out.release()
print("Saved traffic_ttc.mp4")

### Cell 9 — Convert to GIF

In [ ]:
import imageio

reader = imageio.get_reader("traffic_ttc.mp4")
frames = [reader.get_data(i) for i in range(0,120,3)]
imageio.mimsave("traffic_ttc.gif", frames, fps=8)

print("Saved traffic_ttc.gif")

### Cell 10 — Display

In [ ]:
from IPython.display import Video, Image

Video("traffic_ttc.mp4", embed=True)
Image("traffic_ttc.gif")